<a href="https://colab.research.google.com/github/V1LL4pro/Deep-Learning/blob/Corte-1/T3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install keras-tuner

import numpy as np
import os
import gc
from PIL import Image
from tqdm import tqdm
import shutil
import requests
import urllib.parse
import tensorflow as tf
from tensorflow import keras
from keras import layers, backend as K
from sklearn.model_selection import train_test_split
from keras_tuner import RandomSearch
import matplotlib.pyplot as plt

# 1. Descarga Dataset

> Google quickdraw dataset



In [2]:
# Configuración inicial
BASE_PATH = "/content/quickdraw_dataset"
MAX_IMAGES_PER_CLASS = 10000
continuar_proceso = True  # Descarga por defecto

# Verificar existencia del directorio
if os.path.exists(BASE_PATH):
    respuesta = input(f"⚠️ El directorio {BASE_PATH} ya existe. ¿Borrar y recrear? [y/N]: ")
    if respuesta.lower() == 'y':
        print(f"🗑️ Eliminando {BASE_PATH}...")
        shutil.rmtree(BASE_PATH)
    else:
        print("🚨 Proceso cancelado. Usando directorio existente.")
        continuar_proceso = False  # Desactivar la descarga

# Descarga
if continuar_proceso:
    print("\n🚀 Iniciando proceso de descarga...")

    # Obtener categorías
    categories_url = "https://raw.githubusercontent.com/googlecreativelab/quickdraw-dataset/master/categories.txt"
    response = requests.get(categories_url)
    original_categories = [line.strip() for line in response.text.split('\n') if line.strip()]

    # Crear estructura de directorios
    os.makedirs(BASE_PATH, exist_ok=True)
    for category in original_categories:
        safe_name = category.replace(" ", "_")
        os.makedirs(os.path.join(BASE_PATH, safe_name), exist_ok=True)

    def process_category(original_name):
        try:
            url_name = urllib.parse.quote(original_name)
            dir_name = original_name.replace(" ", "_")

            url = f"https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{url_name}.npy"
            response = requests.get(url, stream=True)
            response.raise_for_status()

            temp_path = os.path.join(BASE_PATH, f"{dir_name}.npy")
            with open(temp_path, "wb") as f:
                f.write(response.content)

            images = np.load(temp_path)
            np.random.shuffle(images)

            for i, img_data in enumerate(images[:MAX_IMAGES_PER_CLASS]):
                img = Image.fromarray(img_data.reshape(28, 28).astype(np.uint8))
                img.save(os.path.join(BASE_PATH, dir_name, f"{dir_name}_{i}.png"))

            os.remove(temp_path)

        except Exception as e:
            print(f"❌ Error en {original_name}: {str(e)}")

    # Procesar categorías
    for category in tqdm(original_categories, desc="Procesando"):
        process_category(category)

    print("\n🎉 ¡Proceso completado con éxito!")

⚠️ El directorio /content/quickdraw_dataset ya existe. ¿Borrar y recrear? [y/N]: Y
🗑️ Eliminando /content/quickdraw_dataset...

🚀 Iniciando proceso de descarga...


Procesando: 100%|██████████| 345/345 [58:20<00:00, 10.15s/it]


🎉 ¡Proceso completado con éxito!



# 2. Carga y preparación de datos


> train - 80%, val - 10%, test - 10%

In [ ]:
def load_dataset(base_path, num_classes=345, max_per_class=MAX_IMAGES_PER_CLASS):
    images = []
    labels = []
    class_names = sorted(os.listdir(base_path))

    for label_idx, class_name in enumerate(class_names[:num_classes]):
        class_path = os.path.join(base_path, class_name)
        print(f"Cargando clase #{label_idx+1} {class_name}...")

        for i, img_file in enumerate(os.listdir(class_path)[:max_per_class]):
            img_path = os.path.join(class_path, img_file)
            img = keras.preprocessing.image.load_img(
                img_path, color_mode='grayscale', target_size=(28, 28))
            img_array = keras.preprocessing.image.img_to_array(img)
            images.append(img_array)
            labels.append(label_idx)

    return np.array(images), np.array(labels)

# Cargar datos
X, y = load_dataset('/content/quickdraw_dataset', num_classes=345, max_per_class=MAX_IMAGES_PER_CLASS)

# Preprocesamiento
X = X / 255.0  # Normalización
y = keras.utils.to_categorical(y)  # One-hot encoding

# Dividir datos
# Primera división: 80% entrenamiento, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,          # 20% para validación + test
    random_state=666        # Semilla aleatoria demoniaca
)

# Segunda división: 10% validación, 10% test (del 20% temporal)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,          # Mitad del 20% (10% total)
    random_state=666        # Semilla aleatoria demoniaca
)

Cargando The_Eiffel_Tower...
Cargando The_Great_Wall_of_China...
Cargando The_Mona_Lisa...
Cargando aircraft_carrier...
Cargando airplane...
Cargando alarm_clock...
Cargando ambulance...
Cargando angel...
Cargando animal_migration...
Cargando ant...
Cargando anvil...
Cargando apple...
Cargando arm...
Cargando asparagus...
Cargando axe...
Cargando backpack...
Cargando banana...
Cargando bandage...
Cargando barn...
Cargando baseball...
Cargando baseball_bat...
Cargando basket...
Cargando basketball...
Cargando bat...
Cargando bathtub...
Cargando beach...
Cargando bear...
Cargando beard...
Cargando bed...
Cargando bee...
Cargando belt...
Cargando bench...
Cargando bicycle...
Cargando binoculars...
Cargando bird...
Cargando birthday_cake...
Cargando blackberry...
Cargando blueberry...
Cargando book...
Cargando boomerang...
Cargando bottlecap...
Cargando bowtie...
Cargando bracelet...
Cargando brain...
Cargando bread...
Cargando bridge...
Cargando broccoli...
Cargando broom...
Cargando buck

# 3. Construcción del modelo base


> Capas ocultas: 2 - 4

> Neuronas: 512 - 2048, pasos de 512

> LR: 0.1, 0.01, 0.001, 0.0001



In [ ]:
def build_model(hp=None):
    model = keras.Sequential()

    model.add(keras.Input(shape=(28, 28, 1)))  # Forma de entrada especificada aquí
    model.add(layers.Flatten())

    if hp:
        num_layers = hp.Int('num_layers', 2, 4)  # Entre 2 y 4 capas
        learning_rate = hp.Choice('learning_rate', [1e-1, 1e-2, 1e-3, 1e-4])

        # Definir unidades independientes para cada posible capa
        units = []
        for i in range(num_layers):
            units.append(
                hp.Int(f'units_{i}',
                       min_value=512,
                       max_value=2048,
                       step=512)
            )

        # Añadir solo el número de capas seleccionado
        for i in range(num_layers):
            model.add(layers.Dense(units[i], activation='relu'))

    else:
        # Configuración por defecto (3 capas con diferentes unidades)
        num_layers = 3
        units = [2048, 1024, 512]  # Unidades decrecientes
        learning_rate = 1e-2

        for u in units:
            model.add(layers.Dense(u, activation='relu'))

    model.add(layers.Dense(345, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# 4. Búsqueda de hiperparámetros

> 40 intentos para encontrar los mejores hiperparametros para val_accuracy

> EarlyStop con tolerancia de 3 epocas para evitar overfitting

> Epocas: 20

> Lote: 256







In [ ]:
class MemoryCleaner(keras.callbacks.Callback):
    def on_train_end(self, logs=None):
        K.clear_session()
        gc.collect()

tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=40,
    executions_per_trial=1,
    directory='tuner_dir',
    project_name='quickdraw_mlp'
)

tuner.search(
    X_train, y_train,
    epochs=20,
    validation_data=(X_val, y_val),
    batch_size=256,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3),
        MemoryCleaner()
    ]
)

# 5. Entrenamiento del mejor modelo

> EarlyStop con tolerancia de 5 epocas para evitar overfitting

> Epocas: 100

> Lote: 256

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]
history = best_model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_val, y_val),
    batch_size=256,
    callbacks=[keras.callbacks.EarlyStopping(patience=5)]
)

# Guardar modelo
best_model.save('best_mlp_model.h5')

# 6. Resultados

In [ ]:
# Prediccion con test
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print(f"\nExactitud con test-10%: {test_acc:.4f}")

# Grafica
plt.figure(figsize=(12, 4))

# Gráfico de exactitud
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Exactitud durante el entrenamiento')
plt.ylim(0, 1)  # Establecer rango 0-100%
plt.yticks(np.arange(0, 1.1, 0.1))  # Marcas cada 10%
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{int(y*100)}%'))  # Formato porcentual
plt.legend()

# Gráfico de pérdida
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida durante el entrenamiento')
plt.legend()

plt.tight_layout()
plt.show()